# 03 无监督聚类实验

> 北京交通大学《机器学习与Python编程》研究性专题 — 裂纹图像识别系统
> 负责：蔡昊伭

本 Notebook 包含：
1. 特征提取与标准化
2. 核心聚类方法（K-Means、GMM、DBSCAN）
3. 补充聚类方法（Agglomerative、谱聚类）
4. 聚类效果可视化（PCA/t-SNE 降维展示）
5. 各方法聚类质量对比分析

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# 添加项目根目录到 sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_ROOT
from src.data_utils import (
    apply_clahe,
    apply_median_filter,
    load_dataset,
    split_dataset,
)
from src.evaluation.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from src.models.traditional import extract_features
from src.models.unsupervised import (
    AgglomerativeClusterer,
    DBSCANClusterer,
    GMMClusterer,
    KMeansClusterer,
    SpectralClusterer,
    UnsupervisedPipeline,
)
from src.plot_config import set_chinese_font

set_chinese_font()

## 1. 数据加载与特征提取

加载原始图像，应用预处理（CLAHE + 中值滤波），提取多特征向量。

In [ ]:
# 加载数据集（可调整 max_samples 限制数据量以加速实验）
MAX_SAMPLES = 2000  # 设为 None 加载全部
print("正在加载数据集...")
images, labels = load_dataset(DATA_ROOT, max_samples=MAX_SAMPLES)
print(f"加载完成：{len(images)} 张图像")
print(f"  正样本（有裂纹）：{np.sum(labels == 1)}")
print(f"  负样本（无裂纹）：{np.sum(labels == 0)}")

In [ ]:
# 图像预处理：CLAHE 增强 + 中值滤波去噪
print("正在预处理图像...")
images_processed = np.array([
    apply_median_filter(apply_clahe(img))
    for img in images
])
print(f"预处理完成：{len(images_processed)} 张")

In [ ]:
# 特征提取：HOG + LBP + GLCM + 边缘密度
print("正在提取特征（可能需要几分钟）...")
from tqdm.auto import tqdm

feature_list = []
for img in tqdm(images_processed, desc="特征提取"):
    feature_list.append(extract_features(img))
X = np.array(feature_list, dtype=np.float64)

print(f"特征矩阵形状：{X.shape}")
print(f"特征维度：{X.shape[1]}")

In [ ]:
# 标准化（对 K-Means、GMM、谱聚类等距离/概率方法很重要）
# DBSCAN 在标准化后的 Euclidean 空间中也能更好工作
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"标准化后的特征矩阵：均值={X_scaled.mean():.6f}, 标准差={X_scaled.std():.6f}")

## 2. 核心聚类方法

### 2.1 K-Means 聚类

基于中心的硬划分方法，最小化簇内平方和。适合球形簇，计算效率高。

In [ ]:
# K-Means 聚类
kmeans = KMeansClusterer(n_clusters=2, random_state=42)
kmeans.fit(X_scaled)
kmeans_labels = kmeans.labels_

print(f"K-Means 聚类结果：")
print(f"  簇 0 样本数：{np.sum(kmeans_labels == 0)}")
print(f"  簇 1 样本数：{np.sum(kmeans_labels == 1)}")
print(f"  轮廓系数：{silhouette_score(X_scaled, kmeans_labels):.4f}")
print(f"  ARI (vs 真实标签)：{adjusted_rand_score(labels, kmeans_labels):.4f}")
print(f"  NMI (vs 真实标签)：{normalized_mutual_info_score(labels, kmeans_labels):.4f}")

# 聚类中心
centers = kmeans.get_centers()
print(f"  聚类中心形状：{centers.shape}")

In [ ]:
# K-Means 结果 PCA 可视化
pca_km = PCA(n_components=2, random_state=42)
X_pca_km = pca_km.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 按聚类标签着色
scatter = axes[0].scatter(X_pca_km[:, 0], X_pca_km[:, 1],
                          c=kmeans_labels, cmap="viridis", alpha=0.6, s=10)
axes[0].set_title(f"K-Means 聚类结果 (PCA)\n轮廓系数={silhouette_score(X_scaled, kmeans_labels):.3f}")
axes[0].set_xlabel(f"PC1 ({pca_km.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca_km.explained_variance_ratio_[1]:.1%})")
plt.colorbar(scatter, ax=axes[0], label="聚类标签")

# 按真实标签着色
scatter2 = axes[1].scatter(X_pca_km[:, 0], X_pca_km[:, 1],
                           c=labels, cmap="coolwarm", alpha=0.6, s=10)
axes[1].set_title("真实标签 (PCA)")
axes[1].set_xlabel(f"PC1 ({pca_km.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca_km.explained_variance_ratio_[1]:.1%})")
cbar = plt.colorbar(scatter2, ax=axes[1])
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["无裂纹", "有裂纹"])

plt.tight_layout()
plt.show()

### 2.2 GMM 聚类（高斯混合模型）

基于概率分布的软聚类方法，每个样本属于各类的概率之和为 1。适合椭圆形簇。

In [ ]:
# GMM 聚类
gmm = GMMClusterer(n_components=2, random_state=42)
gmm.fit(X_scaled)
gmm_labels = gmm.labels_
gmm_proba = gmm.probabilities_

print(f"GMM 聚类结果：")
print(f"  簇 0 样本数：{np.sum(gmm_labels == 0)}")
print(f"  簇 1 样本数：{np.sum(gmm_labels == 1)}")
print(f"  轮廓系数：{silhouette_score(X_scaled, gmm_labels):.4f}")
print(f"  ARI (vs 真实标签)：{adjusted_rand_score(labels, gmm_labels):.4f}")
print(f"  NMI (vs 真实标签)：{normalized_mutual_info_score(labels, gmm_labels):.4f}")
print(f"  概率均值（各成分）：{gmm_proba.mean(axis=0).round(4)}")

In [ ]:
# GMM 软聚类可视化：每点按最大概率着色，透明度反映置信度
pca_gmm = PCA(n_components=2, random_state=42)
X_pca_gmm = pca_gmm.fit_transform(X_scaled)
max_proba = gmm_proba.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter = axes[0].scatter(X_pca_gmm[:, 0], X_pca_gmm[:, 1],
                          c=gmm_labels, cmap="viridis", alpha=0.6, s=10)
axes[0].set_title("GMM 硬聚类结果 (PCA)")
plt.colorbar(scatter, ax=axes[0], label="聚类标签")

# 置信度分布直方图
axes[1].hist(max_proba, bins=50, alpha=0.7, color="steelblue", edgecolor="white")
axes[1].axvline(x=0.5, color="red", linestyle="--", label="阈值 0.5")
axes[1].set_xlabel("最大归属概率")
axes[1].set_ylabel("样本数")
axes[1].set_title(f"GMM 概率置信度分布\n平均置信度={max_proba.mean():.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.3 DBSCAN 聚类

基于密度的聚类方法，可自动发现任意形状的簇并识别噪声点。核心参数 `eps`（邻域半径）和 `min_samples`（核心点最小邻域数）对结果影响很大。

In [ ]:
# DBSCAN 聚类（先用默认参数）
dbscan = DBSCANClusterer(eps=0.8, min_samples=10)
dbscan.fit(X_scaled)
dbscan_labels = dbscan.labels_

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = dbscan.n_noise_

print(f"DBSCAN 聚类结果：")
print(f"  发现的簇数：{n_clusters}")
print(f"  噪声点数量：{n_noise} ({n_noise / len(labels) * 100:.1f}%)")
for c in sorted(set(dbscan_labels)):
    if c != -1:
        print(f"  簇 {c} 样本数：{np.sum(dbscan_labels == c)}")
if n_clusters >= 2:
    mask = dbscan_labels != -1
    if np.sum(mask) > 0:
        print(f"  轮廓系数（排除噪声）：{silhouette_score(X_scaled[mask], dbscan_labels[mask]):.4f}")

In [ ]:
# DBSCAN 不同 eps 参数对比
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
eps_values = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0]

pca_db = PCA(n_components=2, random_state=42)
X_pca_db = pca_db.fit_transform(X_scaled)

for i, eps in enumerate(eps_values):
    ax = axes[i // 3, i % 3]
    db_tmp = DBSCANClusterer(eps=eps, min_samples=10)
    db_tmp.fit(X_scaled)
    n_cl = len(set(db_tmp.labels_)) - (1 if -1 in db_tmp.labels_ else 0)
    n_n = db_tmp.n_noise_

    # 噪声点用黑色
    noise_mask = db_tmp.labels_ == -1
    ax.scatter(X_pca_db[noise_mask, 0], X_pca_db[noise_mask, 1],
               c="black", marker="x", s=30, alpha=0.5, label=f"噪声({n_n})")
    non_noise = ~noise_mask
    if np.sum(non_noise) > 0:
        ax.scatter(X_pca_db[non_noise, 0], X_pca_db[non_noise, 1],
                   c=db_tmp.labels_[non_noise], cmap="viridis", alpha=0.6, s=10)
    ax.set_title(f"eps={eps}, 簇={n_cl}, 噪声={n_n}")
    ax.legend(fontsize=8)

plt.suptitle("DBSCAN eps 参数网格搜索 (PCA)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. 补充聚类方法

### 3.1 Agglomerative 层次聚类

自底向上的凝聚层次聚类，通过树状图展示数据的层次结构。

In [ ]:
# Agglomerative 聚类 — 对比不同链接方式
linkages = ["ward", "complete", "average", "single"]
agg_results = {}

for linkage in linkages:
    agg = AgglomerativeClusterer(n_clusters=2, linkage=linkage)
    agg.fit(X_scaled)
    agg_labels = agg.labels_
    agg_results[linkage] = {
        "labels": agg_labels,
        "silhouette": silhouette_score(X_scaled, agg_labels),
        "ari": adjusted_rand_score(labels, agg_labels),
        "nmi": normalized_mutual_info_score(labels, agg_labels),
    }
    print(f"{linkage:>10}: 簇0={np.sum(agg_labels == 0)}, "
          f"簇1={np.sum(agg_labels == 1)}, "
          f"轮廓系数={agg_results[linkage]['silhouette']:.4f}, "
          f"ARI={agg_results[linkage]['ari']:.4f}")

In [ ]:
# Agglomerative 各 linkage 可视化
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, linkage in enumerate(linkages):
    ax = axes[i]
    ax.scatter(X_pca_db[:, 0], X_pca_db[:, 1],
               c=agg_results[linkage]["labels"], cmap="viridis", alpha=0.6, s=10)
    ax.set_title(f"{linkage}\nARI={agg_results[linkage]['ari']:.3f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.suptitle("Agglomerative 不同链接方式对比 (PCA)", fontsize=14)
plt.tight_layout()
plt.show()

### 3.2 谱聚类（Spectral Clustering）

基于图论的聚类方法，将数据映射到 Laplacian 特征空间后聚类，适合非凸形状的簇。

In [ ]:
# 谱聚类
spectral = SpectralClusterer(n_clusters=2, random_state=42)
spectral.fit(X_scaled)
spectral_labels = spectral.labels_

print(f"谱聚类结果：")
print(f"  簇 0 样本数：{np.sum(spectral_labels == 0)}")
print(f"  簇 1 样本数：{np.sum(spectral_labels == 1)}")
print(f"  轮廓系数：{silhouette_score(X_scaled, spectral_labels):.4f}")
print(f"  ARI：{adjusted_rand_score(labels, spectral_labels):.4f}")
print(f"  NMI：{normalized_mutual_info_score(labels, spectral_labels):.4f}")

## 4. 统一 Pipeline 对比

使用 `UnsupervisedPipeline` 一键运行所有 5 种方法并生成对比报告。

In [ ]:
# 一键运行所有聚类方法
pipeline = UnsupervisedPipeline()
results = pipeline.run_all(X_scaled, y_true=labels)

# 打印摘要
print(pipeline.summary())

In [ ]:
# 详细对比表
comparison_data = []
for name, result in results.items():
    m = result["metrics"]
    row = {
        "方法": name,
        "簇数": result["n_clusters"],
        "轮廓系数": round(m.get("silhouette", float("nan")), 4),
        "DB指数": round(m.get("davies_bouldin", float("nan")), 4),
        "CH指数": round(m.get("calinski_harabasz", float("nan")), 2),
        "噪声点": m.get("n_noise", 0),
    }
    if "ari" in m:
        row["ARI"] = round(m["ari"], 4)
        row["NMI"] = round(m["nmi"], 4)
    comparison_data.append(row)

df = pd.DataFrame(comparison_data)
df.set_index("方法", inplace=True)
df

In [ ]:
# 各方法标签 vs 真实标签可视化（t-SNE 降维）
# t-SNE 计算量较大，可先用 PCA 降维到 50 维再跑 t-SNE
print("正在计算 t-SNE...")
X_pca50 = PCA(n_components=50, random_state=42).fit_transform(X_scaled)
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_pca50)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
method_names = list(results.keys()) + ["true"]
all_labels = [results[k]["labels"] for k in results] + [labels]

for i, (name, lab) in enumerate(zip(method_names, all_labels)):
    ax = axes[i // 3, i % 3]
    if name == "true":
        scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                            c=lab, cmap="coolwarm", alpha=0.6, s=10)
        ax.set_title("真实标签")
    else:
        m = results[name]["metrics"]
        ari_str = f"ARI={m.get('ari', float('nan')):.3f}" if "ari" in m else ""
        scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                            c=lab, cmap="viridis", alpha=0.6, s=10)
        ax.set_title(f"{name}\n{ari_str}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("各方法聚类结果 t-SNE 可视化", fontsize=14)
plt.tight_layout()
plt.show()

## 5. 小结

### 各方法特点

| 方法 | 类型 | 核心思想 | 优势 | 局限 |
|------|------|----------|------|------|
| K-Means | 核心 | 中心划分 | 快速，适合球形簇 | 需预设 k，对噪声敏感 |
| GMM | 核心 | 概率分布 | 软聚类，适配椭圆形簇 | 需预设成分数，收敛到局部最优 |
| DBSCAN | 核心 | 密度 | 任意形状，自动识别噪声 | 参数敏感，高维稀疏数据效果差 |
| Agglomerative | 补充 | 层次 | 可探索层次结构 | 计算量大，O(n³) |
| Spectral | 补充 | 图论 | 适合非凸簇 | 计算量大，需预设 k |

### 关键发现

- 裂纹图像的 HOG+LBP+GLCM+边缘密度特征在高维空间中表现出一定的聚类结构
- K-Means 和 GMM 作为线性划分方法，在特征空间中的表现取决于特征的区分能力
- DBSCAN 的噪声检测能力对于识别"难以分类"的边界样本有价值
- 轮廓系数和 ARI 分别从内部/外部角度评价聚类质量，两者结合使用更全面